[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/15_adversarial_search/notebooks/aplicaciones/04_nim.ipynb)

# Aplicación: Nim y la Teoría de Juegos Combinatorios

Este notebook es completamente autocontenido. No necesitas haber leído ningún otro material — solo Python y curiosidad.

## ¿Qué haremos?

1. **Aprender Nim**: reglas, ejemplos y por qué es tan estudiado.
2. **Implementar Minimax** para Nim y usarlo como referencia.
3. **Descubrir el patrón** que hay detrás de las posiciones ganadoras.
4. **Derivar el Nim-Sum** (XOR): la fórmula matemática que resuelve Nim instantáneamente.
5. **Verificar** que la estrategia XOR es equivalente a Minimax.
6. **Hacer benchmarks**: cuánto más rápido es XOR vs Minimax.
7. **(Opcional)** Echar un vistazo al Teorema de Sprague-Grundy.

## ¿Por qué Nim?

Nim parece un juego de niños, pero esconde una estructura matemática profunda. Es el punto de partida de la **teoría de juegos combinatorios** — un campo que demuestra que cualquier juego de dos jugadores, sin información oculta ni azar, puede reducirse a Nim. Eso no es una exageración: es un teorema demostrado.

El recorrido de este notebook es un ejemplo perfecto del método científico aplicado a matemáticas: **observar un patrón → formular una hipótesis → demostrarla → verificarla computacionalmente**.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random
import time
from functools import lru_cache

print("Dependencias listas.")

In [ ]:
# ── Estilo y colores ──────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {
    "primary":    "#2563EB",
    "secondary":  "#10B981",
    "accent":     "#F59E0B",
    "danger":     "#EF4444",
    "win":        "#22C55E",
    "lose":       "#EF4444",
    "neutral":    "#94A3B8",
}

plt.rcParams.update({
    "figure.dpi":     120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size":      10,
    "legend.fontsize":9,
})

np.random.seed(42)
random.seed(42)
print("Estilo configurado.")

---
## Parte 1: El Juego de Nim

### Las reglas

Nim se juega con **pilas de objetos** (fichas, fósforos, piedras). Las reglas son:

1. Hay $k$ pilas, con $a_1, a_2, \ldots, a_k$ objetos respectivamente.
2. Los jugadores se turnan. En cada turno, el jugador elige **una** pila y retira **al menos uno** y hasta **todos** los objetos de esa pila.
3. El jugador que toma el **último objeto gana** (versión *normal play*).

### Ejemplo con una pila

**Nim(7)**: una pila con 7 fichas. Si puedes tomar 1, 2, o 3... ¿ganás o perdés si vas primero?

Hmm, pero en Nim clásico puedes tomar **cualquier cantidad**. Con 1 pila el análisis es trivial: el primer jugador toma todas y gana. La gracia aparece con múltiples pilas.

### Ejemplo con dos pilas: Nim(3, 5)

Hay dos pilas: una con 3 fichas, otra con 5.

```
Pila A: ● ● ●        (3 fichas)
Pila B: ● ● ● ● ●   (5 fichas)
```

¿Existe una estrategia para el primer jugador que garantice ganar, pase lo que pase?

### El estado del juego

Un estado de Nim es una tupla de enteros no negativos — los tamaños de las pilas. El estado terminal es cuando todas las pilas están vacías: $(0, 0, \ldots, 0)$.

El jugador que **enfrenta** el estado $(0, 0, \ldots, 0)$ no puede mover y **pierde**.

In [ ]:
# ── Clase Nim ─────────────────────────────────────────────────────────────────

class Nim:
    """
    Juego de Nim con k pilas.

    Estado: tupla de enteros (a_1, a_2, ..., a_k) — tamaños de pilas.
    Jugadores: MAX (quiere ganar) y MIN (quiere que MAX pierda).
    Utilidad: +1 si MAX gana, -1 si MIN gana.

    En Nim normal-play, el jugador que toma el último objeto GANA.
    Por tanto, el jugador que enfrenta el estado (0,0,...,0) PIERDE.
    """

    def __init__(self, pilas):
        """
        pilas: lista de enteros indicando el tamaño inicial de cada pila.
        Ejemplo: Nim([3, 5]) → dos pilas de 3 y 5.
        """
        self.pilas_iniciales = tuple(pilas)
        self.k = len(pilas)

    def estado_inicial(self):
        """Retorna el estado inicial (tupla de tamaños)."""
        return self.pilas_iniciales

    def jugador(self, s):
        """
        ¿Quién juega en el estado s?
        Alternan: MAX empieza. Para saber el turno, contamos la
        cantidad total de fichas retiradas (paridad).

        Truco: total_inicial - total_actual = fichas retiradas.
        Si fichas retiradas es par → turno de MAX.
        """
        total_actual = sum(s)
        total_inicial = sum(self.pilas_iniciales)
        fichas_retiradas = total_inicial - total_actual
        return 'MAX' if fichas_retiradas % 2 == 0 else 'MIN'

    def acciones(self, s):
        """
        Acciones disponibles desde el estado s.
        Retorna lista de (pila_idx, cantidad): quitar 'cantidad' fichas de pila_idx.
        """
        acciones = []
        for i, n in enumerate(s):
            for cantidad in range(1, n + 1):  # quitar 1, 2, ..., n fichas
                acciones.append((i, cantidad))
        return acciones

    def resultado(self, s, accion):
        """Nuevo estado tras quitar 'cantidad' fichas de la pila 'pila_idx'."""
        pila_idx, cantidad = accion
        nuevo = list(s)
        nuevo[pila_idx] -= cantidad
        return tuple(nuevo)

    def terminal(self, s):
        """El juego termina cuando todas las pilas están vacías."""
        return all(n == 0 for n in s)

    def utilidad(self, s):
        """
        Utilidad del estado terminal.
        El jugador que enfrenta el tablero vacío PIERDE.
        Si es turno de MAX → MAX pierde → utilidad = -1.
        Si es turno de MIN → MIN pierde → utilidad = +1.
        """
        assert self.terminal(s), "utilidad solo se llama en estados terminales"
        # El jugador que tiene que mover desde s=(0,0,...) pierde
        if self.jugador(s) == 'MAX':
            return -1  # MAX pierde
        else:
            return +1  # MIN pierde, MAX gana

    def mostrar(self, s):
        """Imprime el estado de las pilas visualmente."""
        for i, n in enumerate(s):
            barra = '●' * n
            print(f"  Pila {i}: {barra} ({n})")


# ── Prueba básica ─────────────────────────────────────────────────────────────
nim = Nim([3, 5])
s0 = nim.estado_inicial()
print("Estado inicial de Nim(3, 5):")
nim.mostrar(s0)
print(f"\nJugador en turno: {nim.jugador(s0)}")
print(f"Número de acciones: {len(nim.acciones(s0))}")
print(f"¿Terminal? {nim.terminal(s0)}")
print()

# Jugada de ejemplo
accion = (1, 2)  # quitar 2 fichas de la pila 1
s1 = nim.resultado(s0, accion)
print(f"Tras quitar 2 fichas de pila 1:")
nim.mostrar(s1)
print(f"Jugador en turno: {nim.jugador(s1)}")

In [ ]:
# ── Partida de ejemplo: Nim(3, 5) jugada a mano ───────────────────────────────

def visualizar_nim(s, titulo='', ax=None):
    """
    Visualiza el estado de Nim como barras horizontales.
    """
    crear_fig = ax is None
    if crear_fig:
        fig, ax = plt.subplots(figsize=(6, 2.5))
    else:
        fig = ax.figure

    k = len(s)
    max_pila = max(s) if any(n > 0 for n in s) else 1

    colores_pila = [COLORS['primary'], COLORS['secondary'], COLORS['accent'],
                    COLORS['danger'], '#8B5CF6']

    for i, n in enumerate(s):
        color = colores_pila[i % len(colores_pila)]
        ax.barh(i, n, color=color, alpha=0.8, height=0.6,
                edgecolor='white', linewidth=0.5)
        ax.text(n + 0.1, i, str(n), va='center', fontsize=10, fontweight='bold')

    ax.set_yticks(range(k))
    ax.set_yticklabels([f'Pila {i}' for i in range(k)], fontsize=9)
    ax.set_xlim(0, max_pila + 1.5)
    ax.set_xlabel('Número de fichas')
    ax.grid(axis='x', alpha=0.5)

    if titulo:
        ax.set_title(titulo, fontsize=9, pad=4)

    if crear_fig:
        plt.tight_layout()
        plt.show()

    return fig, ax


# Partida de ejemplo paso a paso
partida_demo = [
    ((3, 5), 'Inicio: Nim(3,5)'),
    ((3, 2), 'MAX quita 3 de pila 1'),
    ((1, 2), 'MIN quita 2 de pila 0'),
    ((1, 1), 'MAX quita 1 de pila 1'),
    ((0, 1), 'MIN quita 1 de pila 0'),
    ((0, 0), 'MAX quita 1 de pila 1 → MIN ya no puede mover'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, (estado, titulo) in zip(axes.flat, partida_demo):
    visualizar_nim(estado, titulo=titulo, ax=ax)

plt.suptitle('Ejemplo de partida Nim(3, 5)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("En este ejemplo, MAX ganó.")
print("¿Fue suerte o hay una estrategia?")

---
## Parte 2: Minimax en Nim

Antes de buscar patrones, implementemos minimax para Nim. Esto nos dará una **respuesta correcta** para cualquier posición — aunque puede ser lento para juegos grandes.

Una observación clave: la función `jugador(s)` en Nim depende de cuántas fichas se han retirado totalmente — no solo del turno. Para minimax, lo que importa es saber si el jugador que debe mover quiere maximizar o minimizar. Una forma más limpia para minimax recursivo es pasar explícitamente el turno.

También usaremos **memoización** (con `@lru_cache`) para evitar recalcular los mismos estados.

In [ ]:
# ── Minimax para Nim con memoización ─────────────────────────────────────────

def minimax_nim(estado, es_max, nim_juego):
    """
    Minimax para Nim con memoización interna.

    Parámetros:
        estado    : tupla de pilas.
        es_max    : True si es turno de MAX.
        nim_juego : instancia de Nim (para acciones/resultado/terminal/utilidad).

    Retorna (valor, mejor_accion).
    """
    # Usamos caché con la clave (estado, es_max)
    return _minimax_nim_inner(estado, es_max, nim_juego)


def _minimax_nim_inner(estado, es_max, juego, _cache=None):
    """Implementación interna con diccionario de memoización."""
    if _cache is None:
        _cache = {}

    # Clave de caché
    clave = (estado, es_max)
    if clave in _cache:
        return _cache[clave]

    # Caso base
    if juego.terminal(estado):
        # El jugador que debe mover desde estado vacío pierde
        val = -1 if es_max else +1
        _cache[clave] = (val, None)
        return val, None

    if es_max:
        mejor_val, mejor_acc = float('-inf'), None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            val, _ = _minimax_nim_inner(sucesor, False, juego, _cache)
            if val > mejor_val:
                mejor_val, mejor_acc = val, accion
        _cache[clave] = (mejor_val, mejor_acc)
        return mejor_val, mejor_acc
    else:
        mejor_val, mejor_acc = float('+inf'), None
        for accion in juego.acciones(estado):
            sucesor = juego.resultado(estado, accion)
            val, _ = _minimax_nim_inner(sucesor, True, juego, _cache)
            if val < mejor_val:
                mejor_val, mejor_acc = val, accion
        _cache[clave] = (mejor_val, mejor_acc)
        return mejor_val, mejor_acc


# Versión limpia con caché por instancia
def minimax_nim(estado, es_max, juego):
    cache = {}
    return _minimax_nim_inner(estado, es_max, juego, cache)


print("Minimax para Nim implementado.")
print()

# Casos de prueba simples
casos = [
    ([1],     "Nim(1)   — fácil: MAX toma 1 y gana"),
    ([2],     "Nim(2)   — MAX toma 2 y gana"),
    ([1, 1],  "Nim(1,1) — MAX toma 1, MIN toma 1, MAX no puede → MIN gana"),
    ([1, 2],  "Nim(1,2) — ¿Posición ganadora o perdedora para MAX?"),
    ([2, 2],  "Nim(2,2) — ¿?"),
    ([3, 5],  "Nim(3,5) — ¿?"),
]

print(f"{'Juego':<12} {'Valor':>8} {'Acción óptima'}")
print("-" * 55)
for pilas, desc in casos:
    nim = Nim(pilas)
    s0 = nim.estado_inicial()
    v, a = minimax_nim(s0, True, nim)
    resultado = 'MAX gana' if v > 0 else 'MIN gana'
    print(f"Nim{str(tuple(pilas)):<8}  {v:>+4}    {resultado:<12}  mejor acción: {a}")

In [ ]:
# ── Traza completa de Nim(1, 2) ───────────────────────────────────────────────
# Visualizamos el árbol de juego para el caso más pequeño no trivial

def traza_nim(estado, es_max, juego, prof=0, max_prof=10):
    """
    Imprime el árbol de minimax con indentación.
    Solo útil para juegos pequeños.
    """
    indent = '  ' * prof
    jugador_str = 'MAX' if es_max else 'MIN'

    if juego.terminal(estado):
        val = -1 if es_max else +1
        print(f"{indent}[TERMINAL] {estado} → valor={val:+d} ({jugador_str} pierde)")
        return

    if prof >= max_prof:
        return

    v, a = minimax_nim(estado, es_max, juego)
    print(f"{indent}[{jugador_str}] estado={estado} → valor={v:+d}, mejor acción={a}")

    for accion in juego.acciones(estado):
        sucesor = juego.resultado(estado, accion)
        v_suc, _ = minimax_nim(sucesor, not es_max, juego)
        marca = ' ← ELEGIDA' if accion == a else ''
        print(f"{indent}  acción={accion} → {sucesor}, valor={v_suc:+d}{marca}")
        traza_nim(sucesor, not es_max, juego, prof + 1)


print("Árbol completo de Nim(1, 2):")
print("(Las acciones son (pila_idx, cantidad) — cuántas fichas quitar de qué pila)")
print()
nim12 = Nim([1, 2])
traza_nim(nim12.estado_inicial(), True, nim12, max_prof=4)

In [ ]:
# ── Agentes para torneos ──────────────────────────────────────────────────────

def agente_random_nim(juego, estado):
    """Elige una acción aleatoria entre las disponibles."""
    return random.choice(juego.acciones(estado))


def agente_minimax_nim(juego, estado):
    """Elige la acción óptima usando minimax."""
    es_max = juego.jugador(estado) == 'MAX'
    _, accion = minimax_nim(estado, es_max, juego)
    return accion


def play_nim(juego, agente1, agente2, verbose=False):
    """
    Juega una partida de Nim completa.
    agente1 es MAX, agente2 es MIN.
    Retorna +1 si gana MAX, -1 si gana MIN.
    """
    estado = juego.estado_inicial()
    turno = 0

    while not juego.terminal(estado):
        es_max = juego.jugador(estado) == 'MAX'
        agente = agente1 if es_max else agente2
        accion = agente(juego, estado)
        estado = juego.resultado(estado, accion)
        turno += 1

        if verbose:
            jugador_str = 'MAX' if es_max else 'MIN'
            pila_idx, cantidad = accion
            print(f"  Turno {turno}: {jugador_str} quita {cantidad} de pila {pila_idx} → {estado}")

    # El que enfrenta el estado vacío pierde
    return juego.utilidad(estado)


def torneo_nim(juego, agente1, agente2, n=200, nombre1='MAX', nombre2='MIN'):
    """Juega n partidas y reporta estadísticas."""
    gana_max = 0
    gana_min = 0
    for _ in range(n):
        r = play_nim(juego, agente1, agente2)
        if r > 0:
            gana_max += 1
        else:
            gana_min += 1
    print(f"\n{'=' * 50}")
    print(f" {nombre1} (MAX) vs {nombre2} (MIN) — {n} partidas")
    print(f"{'=' * 50}")
    print(f" MAX gana: {gana_max:>4}  ({100*gana_max/n:5.1f}%)")
    print(f" MIN gana: {gana_min:>4}  ({100*gana_min/n:5.1f}%)")
    return gana_max, gana_min


nim35 = Nim([3, 5])

print("Torneo en Nim(3, 5):")
torneo_nim(nim35, agente_minimax_nim, agente_random_nim, n=200,
           nombre1='Minimax', nombre2='Random')

In [ ]:
# ── Nodos expandidos vs tamaño del juego ─────────────────────────────────────

def contar_nodos_nim(estado, es_max, juego, cache=None):
    """Cuenta cuántos nodos explora minimax (sin poda)."""
    if cache is None:
        cache = {}
    clave = (estado, es_max)
    if clave in cache:
        return cache[clave]

    if juego.terminal(estado):
        cache[clave] = 1
        return 1

    total = 1
    for accion in juego.acciones(estado):
        sucesor = juego.resultado(estado, accion)
        total += contar_nodos_nim(sucesor, not es_max, juego, cache)

    cache[clave] = total
    return total


# Comparamos la cantidad de nodos para distintos tamaños de Nim(n, n)
print("Nodos explorados por minimax en Nim(a, a) (sin memoización de caminos):")
print(f"{'Nim':<14} {'Nodos':>12} {'Total fichas':>14}")
print("-" * 42)

nodos_list = []
tamanos = range(1, 8)
for n in tamanos:
    nim = Nim([n, n])
    s0 = nim.estado_inicial()
    nodos = contar_nodos_nim(s0, True, nim)
    nodos_list.append(nodos)
    print(f"Nim({n},{n})        {nodos:>12,}         {2*n:>4}")

print()
print("Observación: los nodos crecen exponencialmente con el tamaño del juego.")
print("Para Nim(10, 10) o más, minimax es imprácticamente lento.")

In [ ]:
# ── Gráfica: crecimiento exponencial ─────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tamanos_list = list(tamanos)

axes[0].plot(tamanos_list, nodos_list, 'o-',
             color=COLORS['primary'], linewidth=2, markersize=7)
axes[0].set_xlabel('n en Nim(n, n)')
axes[0].set_ylabel('Nodos explorados')
axes[0].set_title('Crecimiento de nodos (escala lineal)', fontweight='bold')

axes[1].semilogy(tamanos_list, nodos_list, 'o-',
                 color=COLORS['danger'], linewidth=2, markersize=7)
axes[1].set_xlabel('n en Nim(n, n)')
axes[1].set_ylabel('Nodos explorados (escala log)')
axes[1].set_title('Crecimiento de nodos (escala log)', fontweight='bold')

plt.suptitle('Minimax en Nim: crecimiento exponencial', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Parte 3: El Patrón Emerge

Minimax nos da la respuesta correcta para cualquier posición, pero es lento. Sin embargo, podemos usarlo para **explorar** el espacio de estados pequeños y buscar un patrón.

Vamos a clasificar todas las posiciones de Nim con 2 pilas de tamaño entre 0 y 5. Para cada posición $(a, b)$, marcamos si es **ganadora** (MAX puede forzar ganar) o **perdedora** (con juego perfecto de MIN, MAX pierde).

In [ ]:
# ── Tabla de posiciones ganadoras/perdedoras en Nim(a, b) ────────────────────

print("Posiciones ganadoras (+) y perdedoras (-) en Nim(a, b):")
print("(+) = el jugador en turno GANA con juego perfecto")
print("(-) = el jugador en turno PIERDE con juego perfecto")
print()

N_MAX = 6  # pilas de 0 a 5

header = "     " + "  ".join(f" b={b}" for b in range(N_MAX))
print(header)
print("     " + "-" * (N_MAX * 5))

tabla_valores = np.zeros((N_MAX, N_MAX), dtype=int)

for a in range(N_MAX):
    fila = f" a={a} |"
    for b in range(N_MAX):
        nim = Nim([a, b])
        s = nim.estado_inicial()
        if nim.terminal(s):
            # Estado ya terminal: el jugador en turno (MAX) pierde
            val = -1
        else:
            val, _ = minimax_nim(s, True, nim)
        tabla_valores[a, b] = val
        simbolo = '  +' if val > 0 else '  -'
        fila += simbolo
    print(fila)

print()
print("Pistas para encontrar el patrón:")
print("¿Qué tienen en común las posiciones marcadas con '-'?")
print("Mira (0,0), (1,1), (2,2), (3,3)... y también (1,2),(2,1) si las hay.")

In [ ]:
# ── Visualización con mapa de calor: ahora añadimos el valor XOR ──────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Izquierda: solo ganar/perder
colores_tabla = np.where(tabla_valores > 0, 1.0, 0.0)
axes[0].imshow(colores_tabla, cmap='RdYlGn', aspect='equal', vmin=0, vmax=1)

for a in range(N_MAX):
    for b in range(N_MAX):
        simbolo = '+' if tabla_valores[a, b] > 0 else '-'
        color = 'black' if colores_tabla[a, b] > 0.5 else 'white'
        axes[0].text(b, a, simbolo, ha='center', va='center',
                     fontsize=18, fontweight='bold', color=color)

axes[0].set_xlabel('Tamaño de pila B (b)', fontsize=11)
axes[0].set_ylabel('Tamaño de pila A (a)', fontsize=11)
axes[0].set_title('Posiciones de Nim(a,b):\nverde=gana, rojo=pierde', fontsize=11)
axes[0].set_xticks(range(N_MAX))
axes[0].set_yticks(range(N_MAX))

# Derecha: con el valor XOR en cada celda
axes[1].imshow(colores_tabla, cmap='RdYlGn', aspect='equal', vmin=0, vmax=1)

for a in range(N_MAX):
    for b in range(N_MAX):
        xor_val = a ^ b
        color = 'black' if colores_tabla[a, b] > 0.5 else 'white'
        axes[1].text(b, a, str(xor_val), ha='center', va='center',
                     fontsize=14, fontweight='bold', color=color)

axes[1].set_xlabel('Tamaño de pila B (b)', fontsize=11)
axes[1].set_ylabel('Tamaño de pila A (a)', fontsize=11)
axes[1].set_title('Mismo mapa, mostrando $a \\oplus b$ (XOR):\nverde=gana, rojo=pierde', fontsize=11)
axes[1].set_xticks(range(N_MAX))
axes[1].set_yticks(range(N_MAX))

plt.tight_layout()
plt.show()

print("¿Ves el patrón? Las celdas ROJAS (perder) son exactamente las que tienen a XOR b = 0.")
print()
print("Verificación:")
errores = 0
for a in range(N_MAX):
    for b in range(N_MAX):
        xor_val = a ^ b
        gana_xor = xor_val != 0
        gana_mm  = tabla_valores[a, b] > 0
        if gana_xor != gana_mm:
            print(f"  Discrepancia en Nim({a},{b}): XOR={xor_val}, minimax={tabla_valores[a,b]}")
            errores += 1
if errores == 0:
    print(f"  Sin discrepancias en las {N_MAX*N_MAX} posiciones.")
    print(f"  La regla 'pierde si a XOR b = 0' es exactamente correcta.")

---
## Parte 4: El Nim-Sum — La Fórmula detrás del Patrón

El patrón que observamos tiene nombre: **nim-sum** (o suma de Nim). Para $k$ pilas de tamaños $a_1, a_2, \ldots, a_k$:

$$\text{nim-sum} = a_1 \oplus a_2 \oplus \cdots \oplus a_k$$

donde $\oplus$ es el operador XOR bit a bit.

### La regla de la estrategia óptima

> Si nim-sum $= 0$ al inicio de tu turno → **posición perdedora** (con juego perfecto del oponente, pierdes).
> 
> Si nim-sum $\neq 0$ al inicio de tu turno → **posición ganadora**: existe al menos una jugada que lleva el nim-sum a 0.

### ¿Por qué funciona? La demostración por casos

La demostración tiene tres partes:

**Propiedad 1: El estado terminal tiene nim-sum = 0.**

$(0, 0, \ldots, 0)$ tiene nim-sum $= 0 \oplus 0 \oplus \cdots \oplus 0 = 0$. El jugador que enfrenta este estado pierde — correcto, porque nim-sum = 0 corresponde a posición perdedora.

**Propiedad 2: Desde nim-sum $\neq 0$, siempre existe una jugada que lleva nim-sum $= 0$.**

Sea $n = a_1 \oplus a_2 \oplus \cdots \oplus a_k \neq 0$. Sea $d$ el bit más significativo de $n$ (el bit más alto en 1). Entonces existe al menos una pila $a_i$ que tiene el bit $d$ activado. Para esa pila, definimos:

$$a_i' = a_i \oplus n$$

Como $a_i$ tiene el bit $d$ activado y $n$ también, $a_i' = a_i \oplus n < a_i$ (el bit $d$ se cancela, lo que garantiza que el nuevo valor es menor). Entonces es una jugada válida (reducir $a_i$ a $a_i'$). Y la nueva nim-sum es:

$$a_1 \oplus \cdots \oplus a_i' \oplus \cdots \oplus a_k = (a_1 \oplus \cdots \oplus a_k) \oplus a_i \oplus a_i' = n \oplus a_i \oplus (a_i \oplus n) = n \oplus a_i \oplus a_i \oplus n = 0$$

¡La nueva nim-sum es 0!

**Propiedad 3: Desde nim-sum $= 0$, cualquier jugada lleva nim-sum $\neq 0$.**

Si $a_1 \oplus \cdots \oplus a_k = 0$ y cambiamos $a_i$ a $a_i' \neq a_i$, entonces:

$$a_1 \oplus \cdots \oplus a_i' \oplus \cdots \oplus a_k = 0 \oplus a_i \oplus a_i' = a_i \oplus a_i' \neq 0$$

porque $a_i \neq a_i'$ (XOR de un número consigo mismo es 0, pero $a_i \neq a_i'$).

**Conclusión:**

- Cualquier estado con nim-sum $= 0$ lleva a nim-sum $\neq 0$ (propiedad 3). 
- Cualquier estado con nim-sum $\neq 0$ puede llevar a nim-sum $= 0$ (propiedad 2).
- El estado terminal tiene nim-sum $= 0$ (propiedad 1).

Por inducción, el jugador que empieza con nim-sum $= 0$ no puede escapar de nim-sum $= 0$ (el oponente siempre puede volver a nim-sum $= 0$), y eventualmente llega al terminal. $\square$

### Rápido repaso de XOR

XOR ($\oplus$) es la operación lógica "o exclusivo" bit a bit:
- $0 \oplus 0 = 0$
- $1 \oplus 0 = 1$  
- $0 \oplus 1 = 1$
- $1 \oplus 1 = 0$

Propiedades clave:
- $a \oplus 0 = a$
- $a \oplus a = 0$
- $a \oplus b = b \oplus a$ (conmutativa)
- $(a \oplus b) \oplus c = a \oplus (b \oplus c)$ (asociativa)

En Python: `a ^ b`.

In [ ]:
# ── Nim-sum y estrategia XOR ──────────────────────────────────────────────────

def nim_sum(estado):
    """
    Calcula el nim-sum: XOR de todos los tamaños de pilas.

    Ejemplo: nim_sum((3, 5)) = 3 XOR 5 = 011 XOR 101 = 110 = 6.
    """
    resultado = 0
    for pila in estado:
        resultado ^= pila
    return resultado


def estrategia_xor(estado):
    """
    Estrategia óptima para Nim basada en nim-sum.

    Si nim-sum != 0: encuentra la jugada que lleva nim-sum a 0.
    Si nim-sum == 0: posición perdedora, hace la jugada menos costosa.

    Retorna la acción como (pila_idx, cantidad).
    """
    ns = nim_sum(estado)

    if ns == 0:
        # Posición perdedora: cualquier jugada es igual de mala.
        # Hacemos la jugada mínima (quitar 1 de la primera pila no vacía)
        # para darle al oponente la mayor oportunidad de equivocarse.
        for i, pila in enumerate(estado):
            if pila > 0:
                return (i, 1)

    # Posición ganadora: encontrar jugada que lleva nim-sum a 0
    for i, pila in enumerate(estado):
        nuevo_pila = pila ^ ns  # Este sería el nuevo tamaño de la pila i
        if nuevo_pila < pila:  # Solo es válido si reducimos la pila
            return (i, pila - nuevo_pila)

    # No debería llegar aquí si el juego está bien implementado
    for i, pila in enumerate(estado):
        if pila > 0:
            return (i, 1)


# ── Verificación de la lógica XOR ────────────────────────────────────────────
print("Ejemplos de nim-sum (XOR):")
print()
ejemplos_xor = [
    (1, 1),  # 1 XOR 1 = 0 → posición perdedora
    (1, 2),  # 1 XOR 2 = 3 → posición ganadora
    (2, 2),  # 2 XOR 2 = 0 → posición perdedora
    (3, 5),  # 3 XOR 5 = 6 → posición ganadora
    (1, 2, 3),  # 1 XOR 2 XOR 3 = 0 → posición perdedora
    (1, 3, 5),  # 1 XOR 3 XOR 5 = 7 → posición ganadora
]

print(f"{'Estado':<16} {'nim-sum':>10} {'Predicción':>14}")
print("-" * 44)
for pilas in ejemplos_xor:
    ns = nim_sum(pilas)
    prediccion = 'GANA (MAX)' if ns != 0 else 'PIERDE (MAX)'
    # Verificar con minimax
    nim = Nim(list(pilas))
    v_mm, _ = minimax_nim(nim.estado_inicial(), True, nim)
    mm_str = 'GANA (MAX)' if v_mm > 0 else 'PIERDE (MAX)'
    match = '✓' if prediccion == mm_str else '✗'
    print(f"Nim{str(pilas):<12} {ns:>10}   {prediccion:<14} minimax={mm_str} {match}")

print()
print("Bits de los ejemplos:")
print(f"  3 = {3:04b}")
print(f"  5 = {5:04b}")
print(f"  3 XOR 5 = {3^5:04b} = {3^5}  (bit más significativo = bit 2)")

In [ ]:
# ── Traza de la estrategia XOR en acción ─────────────────────────────────────

def partida_con_xor(pilas_inicial, verbose=True):
    """
    Juega una partida de Nim donde MAX usa la estrategia XOR y MIN juega random.
    Muestra el razonamiento XOR en cada jugada de MAX.
    """
    estado = tuple(pilas_inicial)
    turno = 0

    if verbose:
        ns = nim_sum(estado)
        print(f"Estado inicial: {estado}")
        print(f"nim-sum = {ns} ({'posición ganadora para MAX' if ns != 0 else 'posición perdedora para MAX'})")
        print()

    while any(n > 0 for n in estado):
        turno += 1
        es_max = sum(pilas_inicial) % 2 == sum(estado) % 2  # paridad

        # Determinar turno por conteo
        total_inicial = sum(pilas_inicial)
        total_actual  = sum(estado)
        fichas_quitadas = total_inicial - total_actual
        es_max_real = fichas_quitadas % 2 == 0

        if es_max_real:
            # MAX usa estrategia XOR
            accion = estrategia_xor(estado)
            pila_idx, cantidad = accion
            nuevo_estado = list(estado)
            nuevo_estado[pila_idx] -= cantidad
            nuevo_estado = tuple(nuevo_estado)
            ns_nuevo = nim_sum(nuevo_estado)
            if verbose:
                print(f"Turno {turno} [MAX/XOR]: quita {cantidad} de pila {pila_idx}")
                print(f"  Estado: {estado} → {nuevo_estado}")
                print(f"  nim-sum después: {ns_nuevo} (debe ser 0)")
        else:
            # MIN juega random
            nim = Nim(list(estado))
            acciones_disponibles = nim.acciones(estado)
            accion = random.choice(acciones_disponibles)
            pila_idx, cantidad = accion
            nuevo_estado = list(estado)
            nuevo_estado[pila_idx] -= cantidad
            nuevo_estado = tuple(nuevo_estado)
            ns_nuevo = nim_sum(nuevo_estado)
            if verbose:
                print(f"Turno {turno} [MIN/random]: quita {cantidad} de pila {pila_idx}")
                print(f"  Estado: {estado} → {nuevo_estado}")
                print(f"  nim-sum después: {ns_nuevo}")

        estado = nuevo_estado
        if verbose:
            print()

    # Estado terminal: el jugador en turno pierde
    fichas_quitadas_final = sum(pilas_inicial) - sum(estado)
    turno_actual_max = fichas_quitadas_final % 2 == 0
    ganador = 'MIN' if turno_actual_max else 'MAX'
    if verbose:
        print(f"Estado final: {estado} (pilas vacías)")
        print(f"El jugador en turno es {'MAX' if turno_actual_max else 'MIN'} → pierde.")
        print(f"GANADOR: {ganador}")
    return ganador


random.seed(13)
print("Partida ejemplo: Nim(3, 5, 7) — MAX usa estrategia XOR")
print("=" * 55)
ganador = partida_con_xor([3, 5, 7])

---
## Parte 5: Verificación — XOR es Equivalente a Minimax

Ahora que tenemos tanto la estrategia XOR como minimax, podemos verificar computacionalmente que producen **exactamente las mismas decisiones** en todas las posiciones de Nim pequeño.

In [ ]:
# ── Verificación completa para Nim con 3 pilas, tamaño ≤ 5 ───────────────────

print("Verificando que estrategia XOR == Minimax para todos Nim(a,b,c), 0 ≤ a,b,c ≤ 5...")
print()

total = 0
iguales = 0
errores = []

for a in range(6):
    for b in range(6):
        for c in range(6):
            nim = Nim([a, b, c])
            s = nim.estado_inicial()

            if nim.terminal(s):
                continue  # estado vacío, no hay decisión que tomar

            total += 1
            ns = nim_sum(s)

            # Predicción XOR
            gana_xor = ns != 0

            # Minimax
            v_mm, _ = minimax_nim(s, True, nim)
            gana_mm = v_mm > 0

            if gana_xor == gana_mm:
                iguales += 1
            else:
                errores.append((a, b, c, ns, v_mm))

print(f"Total de posiciones no-terminales analizadas: {total}")
print(f"Concordancias: {iguales} / {total}")

if not errores:
    print()
    print("¡Verificación completa! La estrategia XOR y Minimax son equivalentes.")
    print("Para TODAS las posiciones de Nim(a,b,c) con a,b,c ≤ 5:")
    print("  nim-sum = 0  ↔  posición perdedora para el jugador en turno")
    print("  nim-sum ≠ 0  ↔  posición ganadora para el jugador en turno")
else:
    print(f"\nDiscrepancias ({len(errores)}):")
    for a, b, c, ns, v in errores:
        print(f"  Nim({a},{b},{c}): nim-sum={ns}, minimax={v}")

In [ ]:
# ── Agente XOR ───────────────────────────────────────────────────────────────

def agente_xor(juego, estado):
    """Agente que usa la estrategia XOR óptima para Nim."""
    return estrategia_xor(estado)


# Torneos: XOR vs Random y XOR vs Minimax
nim_grande = Nim([5, 7, 11])  # un juego más grande

print("Torneos en Nim(5, 7, 11):")
print(f"nim-sum inicial: {nim_sum((5, 7, 11))} → {'posición ganadora' if nim_sum((5, 7, 11)) != 0 else 'posición perdedora'}")
print()

torneo_nim(nim_grande, agente_xor, agente_random_nim, n=200,
           nombre1='XOR', nombre2='Random')

In [ ]:
# Comparación visual: XOR vs Random desde posición ganadora y perdedora

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

casos_torneo = [
    ([3, 5], 'Nim(3,5) — XOR como MAX (gana)'),
    ([2, 2], 'Nim(2,2) — XOR como MAX (pierde: nim-sum=0)'),
]

for ax, (pilas, titulo) in zip(axes, casos_torneo):
    nim = Nim(pilas)
    ns_ini = nim_sum(tuple(pilas))
    n_torneos = 200

    max_g = 0
    min_g = 0
    for _ in range(n_torneos):
        r = play_nim(nim, agente_xor, agente_random_nim)
        if r > 0:
            max_g += 1
        else:
            min_g += 1

    bars = ax.bar(
        ['XOR (MAX) gana', 'Random (MIN) gana'],
        [max_g, min_g],
        color=[COLORS['win'], COLORS['lose']],
        alpha=0.85, edgecolor='white'
    )
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=11)
    ax.set_title(f'{titulo}\nnim-sum={ns_ini}, n={n_torneos}', fontsize=9)
    ax.set_ylabel('Partidas ganadas')
    ax.set_ylim(0, n_torneos + 20)

plt.suptitle('Estrategia XOR: desde posición ganadora y perdedora', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print("Conclusión: la estrategia XOR es perfecta.")
print("  Desde nim-sum ≠ 0: XOR gana siempre (contra cualquier oponente).")
print("  Desde nim-sum = 0: XOR no puede ganar si el oponente juega perfectamente,")
print("  pero puede ganar si el oponente comete errores.")

---
## Parte 6: Benchmark — ¿Cuánto más Rápido es XOR?

La estrategia XOR tiene complejidad temporal $O(k)$ donde $k$ es el número de pilas. No importa cuántas fichas haya en cada pila.

Minimax tiene complejidad que depende del espacio de estados, que crece exponencialmente con el número de fichas totales.

Veamos la diferencia en la práctica.

In [ ]:
# ── Benchmark: XOR vs Minimax ─────────────────────────────────────────────────

def benchmark_nim(pilas, n_trials=10):
    """
    Compara tiempos de XOR vs Minimax para un juego con 'pilas' dadas.
    Minimax solo se ejecuta si el total de fichas es manejable.
    """
    estado = tuple(pilas)

    # Tiempo XOR
    t0 = time.perf_counter()
    for _ in range(n_trials):
        estrategia_xor(estado)
    t_xor = (time.perf_counter() - t0) / n_trials * 1e6  # microsegundos

    # Tiempo Minimax (solo si es factible)
    total_fichas = sum(pilas)
    t_minimax = None
    if total_fichas <= 15:
        nim = Nim(list(pilas))
        t0 = time.perf_counter()
        minimax_nim(estado, True, nim)
        t_minimax = (time.perf_counter() - t0) * 1e3  # milisegundos

    return t_xor, t_minimax


casos_benchmark = [
    ([1, 2],          'Nim(1,2)'),
    ([2, 3],          'Nim(2,3)'),
    ([3, 4, 5],       'Nim(3,4,5)'),
    ([5, 5, 5],       'Nim(5,5,5)'),
    ([10, 15],        'Nim(10,15)'),
    ([100, 200],      'Nim(100,200)'),
    ([1000, 2000],    'Nim(1k,2k)'),
    ([10**6, 10**9],  'Nim(10^6,10^9)'),
]

print(f"{'Juego':<20} {'XOR (µs)':>12} {'Minimax (ms)':>15}")
print("-" * 50)

xor_times = []
mm_times  = []
nombres   = []

for pilas, nombre in casos_benchmark:
    t_xor, t_mm = benchmark_nim(pilas)
    mm_str = f'{t_mm:.3f}' if t_mm is not None else 'inviable'
    print(f"{nombre:<20} {t_xor:>12.3f} {mm_str:>15}")
    xor_times.append(t_xor)
    mm_times.append(t_mm)
    nombres.append(nombre)

print()
print(f"XOR es O(k) en el número de pilas y O(1) en el tamaño de las pilas.")
print(f"Minimax es exponencial en el total de fichas.")
print(f"Para Nim(10^6, 10^9), XOR termina en microsegundos;")
print(f"Minimax nunca terminaría en tiempo razonable.")

In [ ]:
# ── Gráfica de tiempos ────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: tiempo XOR para todos los casos
x_pos = np.arange(len(casos_benchmark))
axes[0].bar(x_pos, xor_times, color=COLORS['primary'], alpha=0.85, edgecolor='white')
for xi, t in zip(x_pos, xor_times):
    axes[0].text(xi, t + 0.01, f'{t:.2f}µs', ha='center', va='bottom', fontsize=7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([n for _, n in casos_benchmark], rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('Tiempo (microsegundos)')
axes[0].set_title('Tiempo de la Estrategia XOR\n(constante, independiente del tamaño)', fontweight='bold')

# Derecha: comparación solo para los casos manejables (minimax disponible)
casos_factibles = [(n, t_xor, t_mm)
                   for (_, n), t_xor, t_mm in zip(casos_benchmark, xor_times, mm_times)
                   if t_mm is not None]

if casos_factibles:
    nombres_f = [x[0] for x in casos_factibles]
    xor_f     = [x[1] for x in casos_factibles]
    mm_f      = [x[2] * 1000 for x in casos_factibles]  # convertir ms a µs

    x2 = np.arange(len(nombres_f))
    ancho = 0.35
    b1 = axes[1].bar(x2 - ancho/2, xor_f, ancho, label='XOR (µs)',
                     color=COLORS['secondary'], alpha=0.85)
    b2 = axes[1].bar(x2 + ancho/2, mm_f,  ancho, label='Minimax (µs)',
                     color=COLORS['danger'],    alpha=0.85)
    axes[1].set_yscale('log')
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(nombres_f, fontsize=8)
    axes[1].set_ylabel('Tiempo (µs, escala log)')
    axes[1].set_title('XOR vs Minimax (escala log)\nSolo juegos pequeños', fontweight='bold')
    axes[1].legend()

plt.tight_layout()
plt.show()

---
## Parte 7 (Opcional): Teorema de Sprague-Grundy

Esta sección va más allá del material requerido del curso. Es para quienes quieran entender por qué Nim es tan fundamental en la teoría de juegos combinatorios.

### ¿Qué es el valor de Grundy?

El **valor de Grundy** (o nim-valor) de una posición $s$ es:

$$G(s) = \text{mex}\{G(s') : s' \in \text{sucesores}(s)\}$$

donde $\text{mex}$ (mínimo excluido) es el menor entero no-negativo que **no** está en el conjunto.

**Casos base:**
- $G(\text{terminal}) = 0$ (posición perdedora para el jugador en turno).

**Interpretación:**
- $G(s) = 0 \Leftrightarrow$ posición perdedora (el jugador que mueve pierde con juego perfecto).
- $G(s) \neq 0 \Leftrightarrow$ posición ganadora.

Para una sola pila de Nim con $n$ objetos, el valor de Grundy es simplemente $n$. Esto se puede verificar fácilmente: desde una pila de $n$, puedes ir a cualquier pila de tamaño $0, 1, \ldots, n-1$, y el mex de $\{0, 1, \ldots, n-1\}$ es $n$.

### El Teorema de Sprague-Grundy

**Teorema (Sprague 1935, Grundy 1939):** Todo juego combinatorio finito, determinista y de información perfecta es **equivalente a un heap de Nim** de tamaño $G(s)$.

Aún más poderoso: si juegas dos juegos simultáneamente (en cada turno el jugador elige en cuál de los dos juegos mover), el valor de Grundy del juego combinado es:

$$G(s_1, s_2) = G_1(s_1) \oplus G_2(s_2)$$

¡El XOR aparece de nuevo!

Esto significa que para **cualquier juego** de este tipo, si calculas los valores de Grundy, puedes reducir el problema a Nim y aplicar la estrategia XOR.

In [ ]:
# ── Cálculo de valores de Grundy ─────────────────────────────────────────────

def grundy_nim_pila(n, cache=None):
    """
    Valor de Grundy para una sola pila de Nim con n objetos.

    Desde una pila de n, puedes ir a cualquier pila de 0..n-1.
    mex({0, 1, ..., n-1}) = n.

    (Este ejemplo es trivial, pero ilustra el cálculo.)
    """
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]

    if n == 0:
        cache[0] = 0
        return 0

    # Valores de Grundy de los sucesores (pilas de 0 a n-1)
    grundy_sucesores = {grundy_nim_pila(i, cache) for i in range(n)}

    # mex = mínimo entero no-negativo no en el conjunto
    mex = 0
    while mex in grundy_sucesores:
        mex += 1

    cache[n] = mex
    return mex


print("Valores de Grundy para Nim de una pila:")
print()
print(f"{'Pila':>8} {'G(pila)':>10}  Observación")
print("-" * 40)
for n in range(10):
    g = grundy_nim_pila(n)
    obs = 'perder' if g == 0 else 'ganar'
    print(f"{n:>8} {g:>10}  ({obs})")

print()
print("Observación: G(pila de n) = n (trivial para Nim de una pila).")
print("La razón: desde n podemos ir a 0,1,...,n-1, y mex({0,...,n-1}) = n.")

In [ ]:
# ── Sprague-Grundy aplicado a Nim multi-pila ─────────────────────────────────

print("Teorema de Sprague-Grundy para Nim multi-pila:")
print()
print("G(a₁, a₂, ..., aₖ) = G(a₁) ⊕ G(a₂) ⊕ ... ⊕ G(aₖ) = a₁ ⊕ a₂ ⊕ ... ⊕ aₖ")
print("(porque G de una pila de n es simplemente n)")
print()
print("Verificación para varios estados:")
print()

casos_grundy = [
    [3, 5],
    [1, 2, 3],
    [4, 5, 6],
    [7, 11, 13],
]

print(f"{'Estado':<16} {'G via XOR':>12} {'G via minimax':>15} {'¿Igual?':>10}")
print("-" * 56)

for pilas in casos_grundy:
    estado = tuple(pilas)
    g_xor = nim_sum(estado)  # Para Nim: G = nim-sum = XOR

    # Grundy 'verdadero' via minimax: solo nos dice si G=0 o G≠0
    nim = Nim(list(pilas))
    v_mm, _ = minimax_nim(estado, True, nim)
    g_mm_cero = (v_mm <= 0)  # True si G=0 (posición perdedora)
    g_xor_cero = (g_xor == 0)

    igual = '✓' if g_xor_cero == g_mm_cero else '✗'
    print(f"Nim{str(tuple(pilas)):<12} {g_xor:>12} {'perdedora' if g_mm_cero else 'ganadora':>15} {igual:>10}")

print()
print("Nota: Minimax solo nos dice si G=0 o G≠0.")
print("Para calcular el valor exacto de Grundy usamos la recursión mex.")

In [ ]:
# ── Un juego diferente: 'substraction game' ──────────────────────────────────
# En este juego, solo puedes quitar 1, 2 o 3 objetos de la pila.
# Los valores de Grundy son diferentes de Nim puro.

def grundy_substraction(n, max_quitar=3, cache=None):
    """
    Valor de Grundy del 'substraction game' con una pila de n objetos,
    donde se puede quitar entre 1 y max_quitar objetos por turno.
    """
    if cache is None:
        cache = {}
    if n in cache:
        return cache[n]

    if n == 0:
        cache[0] = 0
        return 0

    # Sucesores: quitar 1, 2, ..., min(max_quitar, n) fichas
    sucesores_g = {grundy_substraction(n - k, max_quitar, cache)
                   for k in range(1, min(max_quitar, n) + 1)}

    mex = 0
    while mex in sucesores_g:
        mex += 1

    cache[n] = mex
    return mex


print("Substraction game (puedes quitar 1, 2 o 3 fichas):")
print("Valores de Grundy:")
print()
print(f"{'n':>5} {'G(n)':>8}  Posición")
print("-" * 28)
for n in range(13):
    g = grundy_substraction(n)
    obs = 'PIERDE' if g == 0 else 'gana'
    print(f"{n:>5} {g:>8}  {obs}")

print()
print("¿Ves el patrón? Para max_quitar=3: G(n) = n mod 4.")
print("Pierde si n ≡ 0 (mod 4), gana en cualquier otro caso.")
print()
print("Esto es un ejemplo de juego diferente a Nim, pero con valores de Grundy")
print("bien definidos. El teorema de Sprague-Grundy garantiza que este juego")
print("es 'equivalente' a un heap de Nim de tamaño G(n) = n mod 4.")

In [ ]:
# ── Visualización de valores de Grundy ───────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grundy para Nim puro (trivial: G(n) = n)
n_values = list(range(12))
grundy_nim_vals   = [grundy_nim_pila(n)           for n in n_values]
grundy_sub3_vals  = [grundy_substraction(n, 3)     for n in n_values]
grundy_sub5_vals  = [grundy_substraction(n, 5)     for n in n_values]

# Izquierda: Nim puro vs substraction game
axes[0].plot(n_values, grundy_nim_vals,  'o-', label='Nim puro (quitar cualquier cantidad)',
             color=COLORS['primary'],  linewidth=2, markersize=6)
axes[0].plot(n_values, grundy_sub3_vals, 's-', label='Substraction (max=3)',
             color=COLORS['danger'],   linewidth=2, markersize=6)
axes[0].plot(n_values, grundy_sub5_vals, '^-', label='Substraction (max=5)',
             color=COLORS['secondary'], linewidth=2, markersize=6)
axes[0].set_xlabel('Tamaño de la pila (n)')
axes[0].set_ylabel('Valor de Grundy G(n)')
axes[0].set_title('Valores de Grundy: distintos juegos de una pila', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].set_xticks(n_values)

# Derecha: posiciones perdedoras (G=0) en cada juego
perdedoras_nim = [n for n in range(20) if grundy_nim_pila(n) == 0]
perdedoras_sub3 = [n for n in range(20) if grundy_substraction(n, 3) == 0]
perdedoras_sub5 = [n for n in range(20) if grundy_substraction(n, 5) == 0]

axes[1].scatter(perdedoras_nim,  [3] * len(perdedoras_nim),
                marker='x', s=100, color=COLORS['primary'],  linewidths=2,
                label='Nim puro')
axes[1].scatter(perdedoras_sub3, [2] * len(perdedoras_sub3),
                marker='x', s=100, color=COLORS['danger'],   linewidths=2,
                label='Substraction (max=3)')
axes[1].scatter(perdedoras_sub5, [1] * len(perdedoras_sub5),
                marker='x', s=100, color=COLORS['secondary'], linewidths=2,
                label='Substraction (max=5)')

axes[1].set_yticks([1, 2, 3])
axes[1].set_yticklabels(['Sub(max=5)', 'Sub(max=3)', 'Nim puro'])
axes[1].set_xlabel('n (tamaño de la pila)')
axes[1].set_title('Posiciones perdedoras (G=0) para n=0..19', fontweight='bold')
axes[1].set_xlim(-0.5, 20)
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

print()
print("Patrones en las posiciones perdedoras:")
print(f"  Nim puro:        {perdedoras_nim[:8]}... → solo n=0")
print(f"  Sub(max=3):      {perdedoras_sub3[:8]}... → n ≡ 0 (mod 4)")
print(f"  Sub(max=5):      {perdedoras_sub5[:8]}... → n ≡ 0 (mod 6)")
print()
print("Patrón general: en substraction(max=k), pierde si n ≡ 0 (mod k+1).")

---
## Resumen y Reflexiones

### El viaje que hicimos

1. **Nim** parece un juego simple, pero oculta estructura matemática profunda.

2. **Minimax** resuelve Nim exactamente — pero es exponencialmente lento con el tamaño del juego.

3. **El patrón**: explorar las posiciones ganadoras/perdedoras de Nim(a,b) reveló que las posiciones perdedoras son exactamente las de nim-sum = 0.

4. **La prueba** del Teorema de Sprague-Grundy mostró *por qué* el XOR funciona:
   - El estado terminal tiene nim-sum = 0.
   - Desde nim-sum ≠ 0, siempre existe una jugada que lleva a nim-sum = 0.
   - Desde nim-sum = 0, cualquier jugada lleva a nim-sum ≠ 0.
   Esto garantiza que el jugador con nim-sum ≠ 0 puede forzar ganar.

5. **La estrategia XOR** es $O(k)$ en el número de pilas — instantánea para cualquier tamaño de juego.

6. **(Opcional) Sprague-Grundy** extiende estos resultados a cualquier juego combinatorio.

### Lo que hace Nim especial

Nim es el ejemplo canónico de un **juego perfectamente resuelto**. A diferencia del Ajedrez o el Go, donde necesitamos heurísticas y búsqueda profunda, Nim tiene una solución en forma cerrada: una fórmula de tiempo constante que da la jugada óptima.

Esto ilustra una idea importante en IA: a veces la **estructura matemática del problema** permite soluciones mucho más eficientes que la búsqueda bruta. Encontrar esa estructura es el objetivo de la investigación teórica.

### Preguntas para reflexionar

1. ¿Por qué el XOR aparece en la estrategia óptima de Nim? ¿Tiene algún sentido intuitivo en términos de las propiedades del XOR (conmutatividad, asociatividad, $a \oplus a = 0$)?

2. En la versión **misère** de Nim (el que toma el último objeto **pierde**), ¿cambia la estrategia? Pista: la respuesta es casi lo mismo, con una pequeña modificación.

3. Considera el **substraction game** con max_quitar = 3. ¿Cómo cambiaría la estrategia óptima? ¿Puedes derivarla analíticamente?

4. El Teorema de Sprague-Grundy dice que cualquier juego combinatorio se puede reducir a Nim. ¿Cómo reducirías el juego **Chomp** (chocolate quebradizo) a Nim?

5. En tic-tac-toe (notebook 03), ¿podría aplicarse alguna idea de Sprague-Grundy? ¿Por qué sí o por qué no? (Pista: piensa en la simetría del tablero y en si el juego tiene estructura de suma.)